# Tutorial 0: Choose Your `dkx` Path

This notebook is the entry point for students and first-time users. It does not run a heavy transport solve. Instead, it explains which example to open next, what physics each path covers, and how to verify that the local checkout has the expected scripts.

The code solves radially local drift-kinetic problems. At a high level, the kinetic unknown is the departure from a Maxwellian, and the operator contains streaming, magnetic drift, electric-field drift, and collision terms. The examples below show where to learn each part.

## The Five Most Common User Goals

| Goal | Start with | What you learn |
| --- | --- | --- |
| Run one input file and plot output | `01_cli_outputs_and_plots.ipynb` | CLI, HDF5/NetCDF/NPZ, diagnostics PDF |
| Compute transport matrices | `02_transport_and_autodiff.ipynb` | RHSMode=2/3, particle and heat fluxes |
| Differentiate a residual or solve | `02_transport_and_autodiff.ipynb` | JAX gradients, JVP/VJP, implicit differentiation |
| Compute bootstrap current | `03_bootstrap_redl_and_optimization.ipynb` | `FSABjHatOverRootFSAB2`, Redl comparison |
| Add neoclassical objectives to design | `03_bootstrap_redl_and_optimization.ipynb` | optimization proxies and promotion gates |

In [ ]:
from pathlib import Path

repo = Path.cwd()
if not (repo / "examples").is_dir():
    # If launched from examples/tutorials, move back to the repository root.
    repo = Path.cwd().parents[1]

tutorials = repo / "examples" / "tutorials"
required = [
    tutorials / "01_cli_outputs_and_plots.ipynb",
    tutorials / "02_transport_and_autodiff.ipynb",
    tutorials / "03_bootstrap_redl_and_optimization.ipynb",
    repo / "examples" / "tutorials" / "run_quick_output_and_plot.py",
    repo / "examples" / "vmex_finite_beta" / "compare_qs_paper_dkx_redl.py",
]
missing = [path for path in required if not path.is_file()]
assert not missing, missing
print(f"Found {len(required)} first-path tutorial assets.")

## A Minimal Physics Map

The examples are organized around observables, not implementation details.

- Distribution-function solve: find the kinetic response on a velocity and field-line grid.
- Particle and heat fluxes: integrate velocity-space moments of the solved distribution.
- Parallel flow and bootstrap current: compute flow moments such as `<J.B>/sqrt(<B^2>)`.
- Ambipolar electric field: scan or solve for the radial electric field where ion and electron radial currents balance.
- Sensitivities: use JAX transformations or implicit differentiation to get gradients for scans and optimization.

In [ ]:
paths = {
    "quick_cli": "examples/tutorials/run_quick_output_and_plot.py",
    "transport": "examples/transport/transport_matrix_rhsmode2_and_rhsmode3.py",
    "autodiff": "examples/autodiff/implicit_diff_through_gmres_solve_scheme5.py",
    "bootstrap_redl": "examples/vmex_finite_beta/compare_qs_paper_dkx_redl.py",
    "optimization": "examples/optimization/qa_nfp2_dkx_objectives.py",
}

for label, rel in paths.items():
    path = repo / rel
    print(f"{label:16s} -> {rel:78s} {'OK' if path.is_file() else 'MISSING'}")

## Expected Cost By Learning Step

The first examples are intentionally cheap. Full production grids and SFINCS Fortran v3 overlays are validation tasks, not the first classroom exercise.

In [ ]:
import matplotlib.pyplot as plt

labels = ["CLI", "transport", "autodiff", "bootstrap", "optimization"]
relative_cost = [1, 2, 2, 4, 5]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(labels, relative_cost, color=["#2a6f97", "#468faf", "#61a5c2", "#89c2d9", "#a9d6e5"])
ax.set_ylabel("relative tutorial cost")
ax.set_title("Start cheap, then promote to validation runs")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

## Recommended Next Cell To Run In A Terminal

For a fast end-to-end command that writes output files and a PDF panel in a chosen directory:

```bash
python examples/tutorials/run_quick_output_and_plot.py
```

Then open `01_cli_outputs_and_plots.ipynb` to inspect the output schema and plotting commands.

## Validation Rule

A tutorial plot is an educational diagnostic. A publication or design claim needs the validation gates documented in `docs/validation_matrix.rst`: residual convergence, CPU/GPU agreement, resolution checks, and SFINCS Fortran v3 comparison where the model overlap exists.